# 🧠 NeuralAI — Image LoRA Training on Google Colab (Unsloth Studio)

Fine-tune an **SDXL LoRA** in the signature NeuralAI **"vibe stack"** aesthetic
(dark mode, neon accent lighting, high contrast, cinematic) using
[Unsloth Studio](https://unsloth.ai/docs/new/studio) on a free Colab T4 GPU.

**Workflow (per Unsloth docs):**
1. Install Unsloth + launch Studio
2. Load `stabilityai/stable-diffusion-xl-base-1.0`
3. Build / import the NeuralAI dataset (this notebook generates it)
4. Clean & expand with Data Recipes
5. Train the LoRA (no-code UI or scripted fallback)
6. Export the LoRA → load it in the local sidecar via `NEURALAI_LORA_PATH`

The exported LoRA drops straight into `services/diffusion_engine.py`
(`NeuralAIDiffusion`) which already calls `pipe.load_lora_weights(...)`.


## 1. Install Unsloth Dependencies

Run this on a **Colab GPU runtime** (T4 or better). Installs Unsloth + the
diffusers/PEFT stack needed for SDXL LoRA training and export.


In [ ]:
# Install Unsloth (Colab T4 friendly) + image-training deps
!pip install --quiet --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --upgrade diffusers peft accelerate datasets pillow torchvision

import torch, os
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())


## 2. Launch Unsloth Studio

Installs and starts the Studio server. After this cell runs, click **Open
Unsloth Studio** in the white box (or use the fallback link below it) to get
the no-code UI. You can do steps 3–7 entirely from the UI, or continue with the
scripted cells in this notebook.


In [ ]:
# Launch Unsloth Studio (no-code training UI)
!curl -fsSL https://unsloth.ai/install.sh | sh
!unsloth studio -H 0.0.0.0 -p 8888 > unsloth_studio.log 2>&1 &

# Give the server a moment, then surface the public link
import time, re, subprocess
time.sleep(25)
log = subprocess.run(["cat", "unsloth_studio.log"], capture_output=True, text=True).stdout
# Unsloth prints a clickable Studio URL in the log; show last 20 lines
print("\n".join(log.strip().splitlines()[-20:]))
print("\nIf the link is blocked by an adblocker, scroll down in the Studio UI cell for the fallback box.")


## 3. Load a Model

In the Studio UI: **Load model → `stabilityai/stable-diffusion-xl-base-1.0`**.
Or load it here for the scripted fallback path. SDXL is the base the NeuralAI
LoRA will attach to.


In [ ]:
# Scripted fallback: load SDXL base (used if you skip the UI for training)
BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
if torch.cuda.is_available():
    pipe.to("cuda")
print("Loaded base:", BASE_MODEL)


## 4. Import / Build Training Data

The NeuralAI "vibe stack" dataset is **caption-only** (no real images needed to
learn the style). This cell generates `neuralai_captions.jsonl` with
brand-styled prompts. In Studio you can also **Import** this file, or upload
your own `image.png` + `image.txt` pairs for full pixel supervision.


In [ ]:
import json, random
from pathlib import Path

# NeuralAI brand theme pool — the signature "vibe stack" aesthetic
THEMES = [
    "a sentient AI core glowing in a dark server room",
    "neon-lit cyberpunk city skyline at night",
    "abstract neural network visualized as flowing neon threads",
    "a holographic assistant avatar with cyan rim light",
    "futuristic HUD interface floating in dark space",
    "a lone explorer on a neon alien landscape",
    "quantum computer cooling towers with vapor and blue glow",
    "a robotic owl perched on a glowing data monolith",
    "cosmic brain made of stars and electric veins",
    "minimalist logo mark pulsing with magenta energy",
]
BRAND = ("cinematic dark mode, neon accent lighting, high contrast, "
         "hyper-detailed, 8k, volumetric fog, vibe stack aesthetic, "
         "professional concept art")

out = Path("neuralai_dataset"); out.mkdir(exist_ok=True)
random.seed(42)
rows = []
for i in range(400):
    base = random.choice(THEMES)
    rows.append({"file_name": f"neuralai_{i:04d}.png",
                 "caption": f"{base}, {BRAND}"})

with open(out / "metadata.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")
print(f"Wrote {len(rows)} NeuralAI captions to {out/'metadata.jsonl'}")


## 5. Clean & Refine Dataset (Data Recipes)

In Studio, open **Data Recipes** and point it at `neuralai_dataset/metadata.jsonl`
to auto-expand/clean captions (powered by NVIDIA NeMo Data Designer). The
scripted fallback below just sanity-checks the captions and de-duplicates.


In [ ]:
# Scripted fallback: dedupe + verify captions
seen, clean = set(), []
for line in open(out / "metadata.jsonl"):
    cap = json.loads(line)["caption"]
    if cap not in seen:
        seen.add(cap); clean.append(cap)
print(f"Unique captions: {len(clean)} (from {len(rows)})")
print("Sample:", clean[0])


## 6. Configure & Start Training

**No-code (recommended):** In Studio → **Train**, select the SDXL base, import
`metadata.jsonl`, set LoRA rank `r=16`, `lora_alpha=32`, ~10 epochs, and click
**Start**. Studio shows live loss / GPU util.

**Scripted fallback:** train a text-encoder + UNet LoRA with diffusers+PEFT.


In [ ]:
# Scripted fallback trainer (diffusers + PEFT) — runs if you skip the Studio UI
from peft import LoraConfig, get_peft_model
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
unet = pipe.unet
unet = get_peft_model(unet, LoraConfig(
    r=16, lora_alpha=32, target_modules=["to_q","to_v","to_k","to_out.0"],
    lora_dropout=0.05, bias="none"))
unet.print_trainable_parameters()

# NOTE: A full pixel-level SDXL train loop needs latents + captions from real
# images. For caption-only style learning, use the Studio UI (step 6 no-code)
# which handles dataset→latent wiring automatically. This cell just proves the
# LoRA attaches and is ready to .save_pretrained().
lora_dir = Path("neuralai_sdxl_lora"); lora_dir.mkdir(exist_ok=True)
unet.save_pretrained(lora_dir)
print(f"LoRA scaffold saved to {lora_dir} (attach real images in Studio for full training)")


## 7. Chat / Compare with Trained Model

In Studio, use **Model Arena** to load the base and the fine-tuned LoRA side by
side and compare generations. Scripted fallback below generates one test image
with the LoRA attached (if you trained via Studio, re-load the exported adapter
here first).


In [ ]:
# Test generation with the NeuralAI LoRA (loads exported adapter if present)
test_prompt = "a sentient AI core glowing in a dark server room, cinematic dark mode, neon accent lighting, high contrast, 8k, vibe stack aesthetic"
try:
    pipe.load_lora_weights("neuralai_sdxl_lora")
    print("Loaded NeuralAI LoRA for inference test")
except Exception as e:
    print("No trained LoRA yet (train via Studio step 6):", e)

if torch.cuda.is_available():
    pipe.to("cuda")
img = pipe(test_prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
img.save("neuralai_test.png")
img


## 8. Export & Save Model

In Studio → **Export**, save as **safetensors LoRA** (or GGUF). Download the
folder, then wire it into the local NeuralAI sidecar:

```bash
# On the NeuralAI host (ZO Computer / local)
export NEURALAI_DIFFUSION=1
export NEURALAI_LORA_PATH=/path/to/neuralai_sdxl_lora
```

`services/diffusion_engine.py` already calls `pipe.load_lora_weights(NEURALAI_LORA_PATH)`
so the brand LoRA is applied automatically on every local generation.
